In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Propensity Score Matching — LaLonde Dataset

Replicates the classic LaLonde (1986) / Dehejia & Wahba (1999) analysis.

**Goal:** Estimate the average treatment effect on the treated (ATT) of a job-training program (NSW) on 1978 earnings, using propensity score matching to make the non-experimental control groups (CPS, PSID) comparable to the treated group.

**Datasets**
| File | Description |
|---|---|
| `nswre74_treated.txt` | NSW experimental treated group |
| `nswre74_control.txt` | NSW experimental control group |
| `cps_controls.txt` | CPS non-experimental comparison group |
| `psid_controls.txt` | PSID non-experimental comparison group |

**Variables:** `treat`, `age`, `educ`, `black`, `hispanic`, `married`, `nodegree`, `re74`, `re75`, `re78`

In [ ]:
DATA_DIR = Path("../data/raw")

COLS = ["treat", "age", "educ", "black", "hispanic", "married", "nodegree", "re74", "re75", "re78"]

def load_lalonde(path):
    return pd.read_csv(path, sep=r"\s+", header=None, names=COLS)

nsw_treated  = load_lalonde(DATA_DIR / "nswre74_treated.txt")
nsw_control  = load_lalonde(DATA_DIR / "nswre74_control.txt")
cps_controls = load_lalonde(DATA_DIR / "cps_controls.txt")
psid_controls = load_lalonde(DATA_DIR / "psid_controls.txt")

## Assemble analysis datasets

Two comparison setups follow Dehejia & Wahba (1999):
- **NSW (experimental benchmark):** treated + NSW randomized controls — gives the "true" ATT
- **CPS / PSID (observational):** treated + non-experimental controls — PSM should recover the experimental benchmark

In [ ]:
# Experimental benchmark
nsw = pd.concat([nsw_treated, nsw_control], ignore_index=True)

# Observational datasets (NSW treated vs. non-experimental controls)
cps  = pd.concat([nsw_treated, cps_controls],  ignore_index=True)
psid = pd.concat([nsw_treated, psid_controls], ignore_index=True)

print(f"NSW  : {nsw.shape}  | treated={nsw.treat.sum()}, control={( nsw.treat==0).sum()}")
print(f"CPS  : {cps.shape}  | treated={cps.treat.sum()}, control={(cps.treat==0).sum()}")
print(f"PSID : {psid.shape} | treated={psid.treat.sum()}, control={(psid.treat==0).sum()}")

In [ ]:
nsw.head()

In [ ]:
nsw.describe().T